In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
GenDataSet = pd.read_csv("../resource/GenData_Samples.csv")

# Data already has correct column names from Gen_data.py
# No renaming needed - columns are:
# Sample_ID, Day, VBat_Min_V, VBat_Max_V, Ah_Charge_Ah, Ah_Load_Ah, VPv_Max_V, 
# IPv_Max_A, SOC_Pct, Temp_Max_C, Temp_Min_C, Night_Min, Fault_Label

# Convert Night_Min from hours to minutes if needed (already in proper units)
# GenDataSet["Night_Min"] = GenDataSet["Night_Min"] * 60

# Separate features and label
y = GenDataSet["Fault_Label"]
X = GenDataSet.drop(columns=["Fault_Label", "Sample_ID", "Day"])  # Remove non-feature columns
categorical_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=categorical_cols)

# Save the feature column names for later use in validation
feature_columns = X.columns.tolist()

print("Shape X:", X.shape)
print("Unique labels:", y.unique())
print("\nLabel distribution:")
print(y.value_counts())

In [ ]:
# === MIX REAL DATA INTO TRAINING ===
TrueDataMix = pd.read_csv("../resource/TrueSamples.csv")

# Rename columns to match synthetic format
TrueDataMix = TrueDataMix.rename(columns={
    "VBat Min [V]": "VBat_Min_V", "VBat Max [V]": "VBat_Max_V",
    "AhBat [Ah]": "Ah_Charge_Ah", "AhLoad [Ah]": "Ah_Load_Ah",
    "VPv Max [V]": "VPv_Max_V", "IPv Max [A]": "IPv_Max_A",
    "SOC [%]": "SOC_Pct", "TExt Max [°C]": "Temp_Max_C",
    "TExt Min [°C]": "Temp_Min_C", "Night [h]": "Night_Min",
}, errors='ignore')

# Map labels to match synthetic format
def map_label(label):
    label = str(label).strip()
    if 'Normal' in label: return 'Normal Status'
    elif 'F-01' in label: return 'F-01 Controller Bug'
    elif 'F-02' in label: return 'F-02 Low SoC (Weather)'
    elif 'F-03' in label: return 'F-03 PV Issue'
    elif 'F-04' in label: return 'F-04 Load Oscillation'
    elif 'F-05' in label: return 'F-05 Total Power Loss'
    elif 'F-06a' in label: return 'F-06a Battery Aging Trend'
    elif 'F-06b' in label: return 'F-06b Thermal Risk'
    return label

TrueDataMix['Fault_Label'] = TrueDataMix['Fault_Label'].apply(map_label)

# Keep only columns that exist in GenDataSet
common_cols = [c for c in GenDataSet.columns if c in TrueDataMix.columns]
TrueDataMix = TrueDataMix[common_cols]

# Use 80% of real data for training augmentation
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, _ = next(splitter.split(TrueDataMix, groups=TrueDataMix['Sample_ID']))
real_train = TrueDataMix.iloc[train_idx]

# Merge synthetic + real training data
GenDataSet = pd.concat([GenDataSet, real_train], ignore_index=True)
print(f"Combined training data: {len(GenDataSet)} rows")
print(f"Added {len(real_train)} real rows to training")

# Re-extract X and y from combined dataset
y = GenDataSet["Fault_Label"]
X = GenDataSet.drop(columns=["Fault_Label", "Sample_ID", "Day"])
categorical_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=categorical_cols)
feature_columns = X.columns.tolist()

print("\nCombined label distribution:")
print(y.value_counts())


In [ ]:
def reshape_30day_samples(X, y):
    """
    Reshape data where each 30-day sample becomes a sequence.
    Each 30 consecutive rows (1 30-day sample) becomes 1 sequence of shape (30, num_features)
    """
    Xs, ys = [], []
    
    # Assuming GenDataSet has 30 rows per sample (Sample_ID repeats every 30 days)
    sample_size = 30
    
    for i in range(0, len(X) - sample_size + 1, sample_size):
        sample_X = X[i:i+sample_size]  # 30 days
        sample_y = y.iloc[i+sample_size-1]  # Label from last day of sequence
        
        Xs.append(sample_X.values)
        ys.append(sample_y)
    
    return np.array(Xs), np.array(ys)

X_seq, y_seq = reshape_30day_samples(X, y)

print("Sequence shape (samples, days, features):", X_seq.shape)
print("Labels shape:", y_seq.shape)
print("Unique labels in sequence data:", np.unique(y_seq))

In [ ]:
# Encode labels - now with proper descriptive names from Gen_data.py
le = LabelEncoder()
y_encoded = le.fit_transform(y_seq)

print("Label mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {i}: {label}")

# Normalize features (apply to 2D view, then reshape back)
scaler = StandardScaler()
X_2d = X_seq.reshape(-1, X_seq.shape[-1])  # Flatten to 2D for scaling
X_2d = scaler.fit_transform(X_2d)
X_seq = X_2d.reshape(X_seq.shape)  # Reshape back to 3D

print(f"\nData shapes after encoding and scaling:")
print(f"  X_seq: {X_seq.shape}")
print(f"  y_encoded: {y_encoded.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape[0]} samples of {X_train.shape[1]} days, {X_train.shape[2]} features")
print(f"Test set: {X_test.shape[0]} samples of {X_test.shape[1]} days, {X_test.shape[2]} features")

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

In [ ]:
'''
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_classes=8):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, num_layers=2, dropout=0.3)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use last hidden state
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden_size)
        x = self.dropout(torch.relu(self.fc1(last_hidden)))
        return self.fc2(x)
'''
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=X.shape[1], hidden_size=64, num_classes=8):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])
class ImprovedLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_classes=8, dropout_rate=0.4):
        super().__init__()
        
        # Two-layer LSTM with stronger dropout
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True, dropout=dropout_rate)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size//2, batch_first=True, dropout=dropout_rate)
        
        # Bidirectional processing for better context
        self.attention_weight = nn.Linear(hidden_size//2, 1)
        
        # Dense layers with batch norm and dropout
        self.fc1 = nn.Linear(hidden_size//2, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(32, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out1, _ = self.lstm1(x)
        lstm_out2, _ = self.lstm2(lstm_out1)
        
        # Attention mechanism - weight last 10 timesteps more heavily
        batch_size = lstm_out2.shape[0]
        seq_len = lstm_out2.shape[1]
        
        # Use last timestep
        last_hidden = lstm_out2[:, -1, :]  # (batch, hidden_size//2)
        
        # Dense layers with batch norm
        x = self.fc1(last_hidden)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        
        return self.fc3(x)
num_classes = len(le.classes_)
# Create improved model
model = ImprovedLSTMClassifier(
    input_size=X_train.shape[2], 
    hidden_size=128, 
    num_classes=num_classes,
    dropout_rate=0.4
)

#model = LSTMClassifier(input_size=X_train.shape[2], hidden_size=64, num_classes=num_classes)
print(f"Model created for {num_classes} fault types")
print(model)

In [ ]:
# --- Class weights: penalise under-represented faults ---
from collections import Counter
label_counts = Counter(y_encoded)
total = sum(label_counts.values())
weights = torch.tensor(
    [total / (len(label_counts) * label_counts[i]) for i in range(len(label_counts))],
    dtype=torch.float32
)
print('Class weights:', {le.classes_[i]: f'{w:.2f}' for i, w in enumerate(weights)})

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

EPOCHS = 150
train_losses = []
best_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)

    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if (epoch+1) % 10 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}, LR: {lr_now:.6f}')

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

model.load_state_dict(best_state)
print(f'Best training loss: {best_loss:.4f}')


In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.numpy())
        all_labels.extend(batch_y.numpy())

acc = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
# === LOAD REAL DATA + FINE-TUNE ===
TrueData = pd.read_csv("../resource/TrueSamples.csv")
TrueData = TrueData.rename(columns={
    "VBat Min [V]": "VBat_Min_V", "VBat Max [V]": "VBat_Max_V",
    "AhBat [Ah]": "Ah_Charge_Ah", "AhLoad [Ah]": "Ah_Load_Ah",
    "VPv Max [V]": "VPv_Max_V", "IPv Max [A]": "IPv_Max_A",
    "SOC [%]": "SOC_Pct", "TExt Max [°C]": "Temp_Max_C",
    "TExt Min [°C]": "Temp_Min_C", "Night [h]": "Night_Min",
}, errors='ignore')

def map_true_labels(label):
    label_str = str(label).strip()
    if 'Normal' in label_str: return 'Normal Status'
    elif 'F-01' in label_str: return 'F-01 Controller Bug'
    elif 'F-02' in label_str: return 'F-02 Low SoC (Weather)'
    elif 'F-03' in label_str: return 'F-03 PV Issue'
    elif 'F-04' in label_str: return 'F-04 Load Oscillation'
    elif 'F-05' in label_str: return 'F-05 Total Power Loss'
    elif 'F-06a' in label_str: return 'F-06a Battery Aging Trend'
    elif 'F-06b' in label_str: return 'F-06b Thermal Risk'
    return label_str

TrueData['Fault_Label'] = TrueData['Fault_Label'].apply(map_true_labels)

# Extract features
Ty = TrueData["Fault_Label"]
TX = TrueData.drop(columns=["Fault_Label"], errors='ignore')
cols_to_drop = [c for c in ["Sample_ID", "Day"] if c in TX.columns]
if cols_to_drop: TX = TX.drop(columns=cols_to_drop)
cat_cols = TX.select_dtypes(include=["object"]).columns
if len(cat_cols) > 0: TX = pd.get_dummies(TX, columns=cat_cols)
TX = TX.reindex(columns=feature_columns, fill_value=0)
TX_scaled = scaler.transform(TX)

# Build 30-day sequences
sample_size = 30
seqs, lbls, sids = [], [], []
for sid, group in TrueData.groupby('Sample_ID'):
    if len(group) != sample_size: continue
    start = group.index[0]
    seqs.append(TX_scaled[start:start+sample_size])
    lbls.append(group['Fault_Label'].iloc[0])
    sids.append(sid)

X_real, y_real = np.array(seqs), np.array(lbls)
print(f"Real sequences: {len(X_real)}")

# 80/20 split
np.random.seed(42)
idx = np.random.permutation(len(X_real))
split = int(0.8 * len(X_real))
ft_idx, val_idx = idx[:split], idx[split:]
X_ft, y_ft = X_real[ft_idx], y_real[ft_idx]
X_val, y_val = X_real[val_idx], y_real[val_idx]
print(f"Fine-tune: {len(X_ft)}, Validation: {len(X_val)}")

# === FINE-TUNE: freeze LSTMs, retrain head ===
for name, param in model.named_parameters():
    if 'lstm' in name: param.requires_grad = False

y_ft_enc = le.transform(y_ft)
ft_counts = Counter(y_ft_enc)
ft_total = sum(ft_counts.values())
ft_weights = torch.ones(len(le.classes_))
for i in ft_counts:
    ft_weights[i] = ft_total / (len(ft_counts) * ft_counts[i])
print(f"Real class weights: {ft_weights}")

X_ft_t = torch.tensor(X_ft, dtype=torch.float32)
y_ft_t = torch.tensor(y_ft_enc, dtype=torch.long)
ft_loader = DataLoader(TensorDataset(X_ft_t, y_ft_t), batch_size=16, shuffle=True)
ft_criterion = nn.CrossEntropyLoss(weight=ft_weights)
ft_optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005)

model.train()
for epoch in range(50):
    t_loss = 0
    for bx, by in ft_loader:
        ft_optimizer.zero_grad()
        loss = ft_criterion(model(bx), by)
        loss.backward()
        ft_optimizer.step()
        t_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"  FT epoch {epoch+1}, Loss: {t_loss/len(ft_loader):.4f}")

for p in model.parameters(): p.requires_grad = True
print("Fine-tuning complete!")


In [ ]:
# === EVALUATE ON REAL VALIDATION SET ===
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
X_val_t = torch.tensor(X_val, dtype=torch.float32)
with torch.no_grad():
    preds_enc = torch.argmax(model(X_val_t), dim=1).numpy()

predicted_labels = le.inverse_transform(preds_enc)
y_val_true = y_val

overall_acc = accuracy_score(y_val_true, predicted_labels)
print('='*60)
print(f'VALIDATION ACCURACY (REAL DATA): {overall_acc:.2%}')
print('='*60)

validation_results = pd.DataFrame({
    'Actual_Fault': y_val_true,
    'Model_Prediction': predicted_labels,
    'Match': y_val_true == predicted_labels
})

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_val_true, predicted_labels, labels=le.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Validation Confusion Matrix (After Fine-Tuning)', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

print('\nClassification Report:')
print(classification_report(y_val_true, predicted_labels))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Create Short Names Mapping (for visualization)
short_name_map = {
    'Normal Status': 'Normal',
    'F-01 Controller Bug': 'F-01',
    'F-02 Low SoC (Weather)': 'F-02',
    'F-03 PV Issue': 'F-03',
    'F-04 Load Oscillation': 'F-04',
    'F-05 Total Power Loss': 'F-05',
    'F-06a Battery Aging Trend': 'F-06a',
    'F-06b Thermal Risk': 'F-06b'
}

# Apply mapping (handles multi-labels if they exist)
def shorten_label(label):
    if '|' in str(label):
        parts = label.split('|')
        return "|".join([short_name_map.get(p.strip(), p.strip()[:5]) for p in parts])
    return short_name_map.get(str(label), str(label)[:10])

viz_df = validation_results.copy()
viz_df['Actual_Short'] = viz_df['Actual_Fault'].apply(shorten_label)
viz_df['Pred_Short'] = viz_df['Model_Prediction'].apply(shorten_label)

# 2. Set the visual style
plt.figure(figsize=(16, 6))
sns.set_context("talk")

# Get unique labels in sorted order
labels = sorted(viz_df['Actual_Short'].unique())

# Plot A: Normalized Confusion Matrix (Percentages)
plt.subplot(1, 2, 1)
cm = confusion_matrix(viz_df['Actual_Short'], viz_df['Pred_Short'], labels=labels, normalize='true')

sns.heatmap(cm, annot=True, fmt='.0%', cmap='rocket_r', cbar=False,
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 10})
plt.title('Prediction Accuracy (%)', pad=20)
plt.ylabel('Actual')
plt.xlabel('Predicted')

# Plot B: Prediction Count Comparison
plt.subplot(1, 2, 2)
counts = viz_df.melt(value_vars=['Actual_Short', 'Pred_Short'], var_name='Type', value_name='Fault')
counts['Type'] = counts['Type'].replace({'Actual_Short': 'Actual', 'Pred_Short': 'Predicted'})

sns.countplot(data=counts, x='Fault', hue='Type', palette='viridis', order=labels)
plt.title('Sample Counts', pad=20)
plt.xticks(rotation=0)
plt.xlabel('')
plt.ylabel('')
plt.legend(title=None)

sns.despine()
plt.tight_layout()
plt.show()

# 3. Compact Metrics Summary
print("\n" + "="*50)
print(f"{'FAULT TYPE':<20} | {'ACCURACY':<10}")
print("-" * 50)
for label in labels:
    subset = viz_df[viz_df['Actual_Short'] == label]
    if len(subset) > 0:
        acc = (subset['Actual_Short'] == subset['Pred_Short']).mean()
        count = len(subset)
        print(f"{label:<20} | {acc:>10.1%} ({count} samples)")
print("="*50)